# 04 - Machine Learning

This notebook compares Linear Regression, Random Forest Regression and XGBoost Regression using a spatial block holdout. Performance is evaluated with R2, RMSE and MAE.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.config import PROCESSED_DIR, OUTPUTS_DIR

df = pd.read_parquet(PROCESSED_DIR / "london_pm25_final.parquet")
TARGET = "pm25_mean"
FEATURES = [
    "major_mean_aadf", "major_hgv_mean_aadf", "major_road_length_m",
    "minor_vkm_total", "minor_hgv_vkm", "green_cover_pct",
]
print("Rows:", len(df))
print("Features:", FEATURES)

Rows: 6670
Features: ['major_mean_aadf', 'major_hgv_mean_aadf', 'major_road_length_m', 'minor_vkm_total', 'minor_hgv_vkm', 'green_cover_pct']


In [2]:
RANDOM_STATE = 42
SPATIAL_BLOCK_SIZE = 5000
TEST_BLOCK_FRACTION = 0.20

model_df = df.copy()
model_df["block_x"] = np.floor(model_df["centroid_x"] / SPATIAL_BLOCK_SIZE).astype(int)
model_df["block_y"] = np.floor(model_df["centroid_y"] / SPATIAL_BLOCK_SIZE).astype(int)
model_df["spatial_block"] = model_df["block_x"].astype(str) + "_" + model_df["block_y"].astype(str)
blocks = model_df["spatial_block"].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(blocks)
n_test_blocks = max(1, int(np.ceil(len(blocks) * TEST_BLOCK_FRACTION)))
test_blocks = set(blocks[:n_test_blocks])
train = model_df[~model_df["spatial_block"].isin(test_blocks)].copy()
test = model_df[model_df["spatial_block"].isin(test_blocks)].copy()
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print("Total spatial blocks:", len(blocks))
print("Training rows:", len(train))
print("Testing rows:", len(test))

Total spatial blocks: 91
Training rows: 5464
Testing rows: 1206


In [3]:
def calculate_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": rmse,
        "MAE": mean_absolute_error(y_true, y_pred),
    }

linear_model = LinearRegression().fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)
linear_metrics = calculate_metrics(y_test, linear_pred)

random_forest = RandomForestRegressor(
    n_estimators=400, min_samples_leaf=2, max_features=0.8,
    random_state=RANDOM_STATE, n_jobs=-1,
).fit(X_train, y_train)
rf_pred = random_forest.predict(X_test)
rf_metrics = calculate_metrics(y_test, rf_pred)

xgb = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=5, min_child_weight=3,
    subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror",
    eval_metric="rmse", random_state=RANDOM_STATE, n_jobs=-1,
).fit(X_train, y_train, verbose=False)
xgb_pred = xgb.predict(X_test)
xgb_metrics = calculate_metrics(y_test, xgb_pred)

comparison = pd.DataFrame([
    {"model": "Linear Regression", **linear_metrics},
    {"model": "Random Forest", **rf_metrics},
    {"model": "XGBoost", **xgb_metrics},
]).sort_values("RMSE").reset_index(drop=True)
display(comparison.round(4))

,model,R2,RMSE,MAE
0,XGBoost,0.5984,0.4131,0.3336
1,Random Forest,0.5864,0.4192,0.3394
2,Linear Regression,0.4310,0.4917,0.3863


In [4]:
importance = pd.DataFrame({"feature": FEATURES, "importance": xgb.feature_importances_}).sort_values("importance", ascending=False)
display(importance.round(4))

fig, ax = plt.subplots(figsize=(9, 6))
ordered = importance.sort_values("importance")
ax.barh(ordered["feature"], ordered["importance"])
ax.set_xlabel("XGBoost feature importance")
ax.set_ylabel("Predictor")
ax.set_title("XGBoost Feature Importance")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, xgb_pred, s=16, alpha=0.45)
minimum = min(y_test.min(), xgb_pred.min())
maximum = max(y_test.max(), xgb_pred.max())
ax.plot([minimum, maximum], [minimum, maximum], linestyle="--")
ax.set_xlabel("Observed modelled PM2.5 (µg/m³)")
ax.set_ylabel("Predicted PM2.5 (µg/m³)")
ax.set_title("XGBoost: Observed vs Predicted on Spatial Holdout")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

,feature,importance
2,major_road_length_m,0.4235
3,minor_vkm_total,0.1794
4,minor_hgv_vkm,0.1375
5,green_cover_pct,0.0992
1,major_hgv_mean_aadf,0.0859
0,major_mean_aadf,0.0745


In [5]:
residuals = y_test - xgb_pred
print("Mean residual:", residuals.mean())
print("MAE:", np.abs(residuals).mean())
print("RMSE:", np.sqrt(np.mean(residuals ** 2)))

final_model = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=5, min_child_weight=3,
    subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror",
    eval_metric="rmse", random_state=RANDOM_STATE, n_jobs=-1,
).fit(df[FEATURES], df[TARGET], verbose=False)

full_predictions = df[["grid_id", "centroid_x", "centroid_y", TARGET]].copy()
full_predictions["final_model_prediction"] = final_model.predict(df[FEATURES])
full_predictions["final_model_residual"] = full_predictions[TARGET] - full_predictions["final_model_prediction"]
display(full_predictions.describe())

Mean residual: -0.04364622577190863
MAE: 0.33359351813226185
RMSE: 0.4130841365422036


,centroid_x,centroid_y,pm25_mean,final_model_prediction,final_model_residual
count,6670.000000,6670.000000,6670.000000,6670.000000,6670.000000
mean,531330.584708,179588.605697,8.773117,8.772964,0.000152
std,13561.000151,10360.111495,0.728961,0.625442,0.284025
min,503750.000000,155750.000000,7.033696,7.360711,-1.200998
25%,520750.000000,171750.000000,8.255383,8.343596,-0.176589
50%,531750.000000,180250.000000,8.738196,8.751943,-0.007480
75%,542250.000000,187750.000000,9.248711,9.149130,0.166947
max,561750.000000,200750.000000,11.674392,11.551817,1.746563


## Final model interpretation

XGBoost achieved the strongest predictive performance on the spatial holdout and was selected using held-out RMSE. It was then refitted using all 6,670 usable modelling cells to produce the final London prediction surface. Feature importance indicates model reliance, not causal effect.

In [6]:
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUTPUTS_DIR / "notebook_model_comparison.csv", index=False)
importance.to_csv(OUTPUTS_DIR / "notebook_xgb_feature_importance.csv", index=False)
full_predictions.to_csv(OUTPUTS_DIR / "notebook_final_predictions.csv", index=False)
print("Notebook outputs saved.")

Notebook outputs saved.
